# MovieLens Recommender — Stage 2: Matrix Factorization (PyTorch)

Stage 1 gave us the bar to beat: **Recall@10 = 0.0700, NDCG@10 = 0.0632** (popularity).

Now we build the first *personalized* model. Matrix factorization learns a small vector
(an **embedding**) for every user and every movie:
- A user's vector captures their taste.
- A movie's vector captures its character.
- Predicted affinity = **dot product** of the two vectors. High dot product -> likely to like.

The vectors are *learned* from who-liked-what, so different users get different
recommendations. This is the classic collaborative-filtering idea (the core of the
Netflix Prize). If it can't beat 0.0700, it isn't earning its complexity.

**Runtime: turn ON GPU** (Runtime -> Change runtime type -> GPU). Training uses it.

> This notebook assumes `train`, `test`, and the metric helpers from Stage 1 exist.
> Easiest path: run Stage 1 first in the same session, then run this. (The first cells
> below re-create what's needed in case you're starting fresh — just set the data path.)

## 0. (If starting fresh) reload the data + rebuild the split
If you still have `train`/`test` in memory from Stage 1, you can SKIP this cell.
Otherwise run it to rebuild them.

In [ ]:
import pandas as pd, numpy as np

try:
    train.shape; test.shape   # already in memory?
    print("train/test already loaded — skipping reload")
except NameError:
    import os
    if not os.path.exists('ml-25m/ratings.csv'):
        !wget -q https://files.grouplens.org/datasets/movielens/ml-25m.zip && unzip -q -o ml-25m.zip
    ratings = pd.read_csv('ml-25m/ratings.csv')
    pos = ratings[ratings.rating >= 4.0].copy()
    uc = pos.userId.value_counts(); ic = pos.movieId.value_counts()
    pos = pos[pos.userId.isin(uc[uc>=5].index) & pos.movieId.isin(ic[ic>=5].index)]
    pos = pos.sort_values(['userId','timestamp']).reset_index(drop=True)
    pos['rk'] = pos.groupby('userId').cumcount()
    pos['sz'] = pos.groupby('userId')['userId'].transform('size')
    pos['is_test'] = pos['rk'] >= (pos['sz']*0.8)
    train = pos[~pos.is_test]; test = pos[pos.is_test]
    print(f"rebuilt -> train {len(train):,}  test {len(test):,}")

## 1. Map user/movie IDs to contiguous indices
Embedding tables are indexed 0..N-1, but MovieLens IDs are arbitrary numbers with gaps.
So we map each real userId/movieId to a compact integer index.

In [ ]:
user_ids = train.userId.unique()
item_ids = train.movieId.unique()
u2idx = {u:i for i,u in enumerate(user_ids)}
i2idx = {m:i for i,m in enumerate(item_ids)}
n_users, n_items = len(u2idx), len(i2idx)
print(f"{n_users:,} users, {n_items:,} items")

# training pairs as index arrays
tr = train[train.movieId.isin(i2idx)]
train_u = tr.userId.map(u2idx).to_numpy()
train_i = tr.movieId.map(i2idx).to_numpy()
print("training pairs:", len(train_u))

## 2. The model — two embedding tables + a dot product
That's the entire model: a table of user vectors, a table of item vectors, plus small
per-user and per-item bias terms (which absorb "this user rates everything highly" /
"this movie is universally liked" effects).

In [ ]:
import torch, torch.nn as nn
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("device:", device)

class MF(nn.Module):
    def __init__(self, n_users, n_items, dim=64):
        super().__init__()
        self.user_emb = nn.Embedding(n_users, dim)
        self.item_emb = nn.Embedding(n_items, dim)
        self.user_bias = nn.Embedding(n_users, 1)
        self.item_bias = nn.Embedding(n_items, 1)
        # small init helps training stability
        nn.init.normal_(self.user_emb.weight, std=0.05)
        nn.init.normal_(self.item_emb.weight, std=0.05)
        nn.init.zeros_(self.user_bias.weight)
        nn.init.zeros_(self.item_bias.weight)

    def forward(self, u, i):
        # score = dot(user_vec, item_vec) + user_bias + item_bias
        dot = (self.user_emb(u) * self.item_emb(i)).sum(1)
        return dot + self.user_bias(u).squeeze(1) + self.item_bias(i).squeeze(1)

model = MF(n_users, n_items, dim=64).to(device)
print(model)

## 3. Negative sampling — the key training idea
We only have *positives* ("user liked movie"). But a model trained only on positives
would just say "everything is great." So for each positive pair, we also draw a **random
movie the user did NOT interact with** as a *negative*, and train the model to score the
positive higher than the negative.

This is **Bayesian Personalized Ranking (BPR)**: maximize the gap between a user's real
item and a random unseen item. It directly optimizes *ranking*, which is what recommendation
is about.

In [ ]:
# set of items each user has seen (to avoid sampling a real positive as a "negative")
from collections import defaultdict
seen = defaultdict(set)
for u,i in zip(train_u, train_i):
    seen[u].add(i)

train_u_t = torch.tensor(train_u, dtype=torch.long)
train_i_t = torch.tensor(train_i, dtype=torch.long)

def sample_negatives(u_batch):
    # random items; resample the few that happen to be real positives
    negs = np.random.randint(0, n_items, size=len(u_batch))
    for k,(u,neg) in enumerate(zip(u_batch.numpy(), negs)):
        while neg in seen[u]:
            neg = np.random.randint(0, n_items)
        negs[k] = neg
    return torch.tensor(negs, dtype=torch.long)

## 4. Train with BPR loss
BPR loss = -log(sigmoid(score_positive - score_negative)), averaged over the batch.
Minimizing it pushes each user's real items above random unseen items.

In [ ]:
import torch.nn.functional as F
opt = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=1e-5)
BATCH = 8192
N = len(train_u_t)
EPOCHS = 8

for epoch in range(EPOCHS):
    perm = torch.randperm(N)
    total = 0.0
    model.train()
    for start in range(0, N, BATCH):
        idx = perm[start:start+BATCH]
        u = train_u_t[idx].to(device)
        pos_i = train_i_t[idx].to(device)
        neg_i = sample_negatives(train_u_t[idx]).to(device)

        pos_score = model(u, pos_i)
        neg_score = model(u, neg_i)
        loss = -F.logsigmoid(pos_score - neg_score).mean()

        opt.zero_grad(); loss.backward(); opt.step()
        total += loss.item() * len(idx)
    print(f"epoch {epoch+1}/{EPOCHS}  BPR loss: {total/N:.4f}")

## 5. Recommend + evaluate against the SAME metrics as Stage 1
For a user, score ALL items by dot product with their vector, remove items they saw in
training, and take the top-K. Then we reuse Recall@K / NDCG@K to compare directly to the
popularity baseline (0.0700 / 0.0632).

In [ ]:
model.eval()
item_vecs = model.item_emb.weight.detach()          # (n_items, dim)
item_b = model.item_bias.weight.detach().squeeze(1) # (n_items,)

@torch.no_grad()
def recommend_mf(user_id, k=10):
    if user_id not in u2idx:            # unseen user -> can't personalize
        return []
    ui = u2idx[user_id]
    uvec = model.user_emb.weight[ui]                 # (dim,)
    ub = model.user_bias.weight[ui]
    scores = item_vecs @ uvec + item_b + ub          # score every item
    scores[list(seen[ui])] = -1e9                    # mask training-seen items
    top = torch.topk(scores, k).indices.cpu().numpy()
    # map item indices back to real movieIds
    return [item_ids[t] for t in top]

def dcg(hits): return sum(1.0/np.log2(i+2) for i,h in enumerate(hits) if h)

def evaluate(recommend_fn, k=10, n_users=3000, seed=0):
    test_by_user = test.groupby('userId').movieId.apply(set).to_dict()
    users = list(test_by_user.keys())
    sample = np.random.default_rng(seed).choice(users, size=min(n_users,len(users)), replace=False)
    recalls, ndcgs = [], []
    for u in sample:
        truth = test_by_user[u]
        if not truth: continue
        recs = recommend_fn(u, k)
        if not recs: continue
        hits = [1 if m in truth else 0 for m in recs]
        recalls.append(sum(hits)/min(len(truth),k))
        ideal = dcg([1]*min(len(truth),k))
        ndcgs.append(dcg(hits)/ideal if ideal>0 else 0.0)
    return float(np.mean(recalls)), float(np.mean(ndcgs))

rec, ndcg_score = evaluate(recommend_mf)
print(f"MATRIX FACTORIZATION ->  Recall@10: {rec:.4f}   NDCG@10: {ndcg_score:.4f}")
print(f"(popularity baseline was  Recall@10: 0.0700   NDCG@10: 0.0632)")

## 6. Read the result
If Recall@10 and NDCG@10 are **above the popularity baseline**, personalization is
working — the learned user/item vectors capture taste beyond "what's popular." That
lift over the baseline is the honest headline for this stage.

If it's only *slightly* above (or below), that's a real, interesting finding too — it
often means we need more epochs, a different learning rate, or better negative sampling.
We'll diagnose it rather than hide it.

**Next (Stage 3):** two-tower retrieval — the modern architecture that scales this idea
to fast nearest-neighbor lookup over millions of items.